# Competition notebook: validated Qwen + SVC architecture

This production path is intentionally conservative and deterministic:

1. Validate the official competition schema and checkpoint label order.
2. Fit the four-view SVC on official train plus the fixed `pool_v1` corpus.
3. Run the frozen Qwen3-8B and Qwen3-14B two-stage LoRA checkpoints.
4. Blend only validated probability matrices in log space.
5. Apply optional modules only when an explicit feature flag is enabled.
6. Validate the final parquet before writing it.

Default production flags are: external augmentation **off**, label calibration **off**, sibling specialists **off**, centred twin **off**, and Sinkhorn balancing **on**. Every optional module is isolated so a research experiment cannot silently change the production baseline.

In [ ]:
import gc, glob, json, os, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()
INPUT_DIR = "/kaggle/input"
PIPELINE_VERSION = "baseline_qwen_svc_v1"
# Feature gates: change only in a dedicated research experiment, never implicitly.
ENABLE_EXTERNAL_AUGMENTATION = False
ENABLE_LABEL_CALIBRATION = False
ENABLE_SIBLING_SPECIALIST = False
ENABLE_TWIN = False
ENABLE_SINKHORN = True

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
W_SVC = 1.021
LORAS = {
    "lora8b_v1": dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),
    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
}
TWIN = dict(thresh=0.85, min_size=2, max_size=18)

def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Required Kaggle input not found: {pattern}")
    return hits[0]

def safe_prob(scores):
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim != 2 or scores.shape[1] != K:
        raise ValueError(f"Expected score matrix (_, {K}), got {scores.shape}")
    if not np.isfinite(scores).all():
        raise FloatingPointError("Non-finite model scores")
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(np.clip(scores, -50.0, 50.0))
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    if not np.isfinite(probs).all() or not np.allclose(probs.sum(1), 1.0, atol=1e-5):
        raise FloatingPointError("Invalid probability matrix")
    return probs

def validate_inputs(train, test):
    required_train = {"id", "prompt_id", "text", "model"}
    required_test = {"id", "text"}
    if not required_train.issubset(train.columns):
        raise ValueError(f"train.csv missing columns: {required_train - set(train.columns)}")
    if not required_test.issubset(test.columns):
        raise ValueError(f"test.csv missing columns: {required_test - set(test.columns)}")
    if train.id.isna().any() or test.id.isna().any() or train.id.duplicated().any() or test.id.duplicated().any():
        raise ValueError("IDs must be non-null and unique")
    unknown = sorted(set(train.model.dropna()) - set(LABELS))
    if unknown:
        raise ValueError(f"Unknown train labels: {unknown}")
    if train.model.isna().any():
        raise ValueError("Null labels in train.csv")
    train["text"] = train["text"].fillna("").astype(str)
    test["text"] = test["text"].fillna("").astype(str)
    return train, test

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
train, test = validate_inputs(train, test)
y = train["model"].map(LABEL_TO_ID).to_numpy(dtype=np.int64)
LOGIT = np.zeros((len(test), K), dtype=np.float64)
print(f"pipeline={PIPELINE_VERSION} train={train.shape} test={test.shape} flags={{twin:{ENABLE_TWIN}, calibration:{ENABLE_LABEL_CALIBRATION}, sibling:{ENABLE_SIBLING_SPECIALIST}, sinkhorn:{ENABLE_SINKHORN}}}", flush=True)


## 1. The n gram model

Each model has typing habits. In the training data claude and GLM never use curly quotes, while gpt uses a curly apostrophe in about half its passages. Gemini almost never uses an em dash, while Claude uses one in three passages out of four. Character n grams pick this up.

We look at each passage four ways: raw characters +  words plus punctuation +  the *shape* of the text (`Xxxx xx x`) + function words only, with every content word replaced by `CONTENT` so the topic can't leak in. A linear SVM sits on top, trained on train + the public pool with balanced class weights.


In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]

def shape_text(text):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", text)

def function_text(text):
    return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT" for w in re.findall(r"\w+|[^\w\s]", text))

def fit_svc(train_texts, train_y, test_texts):
    blocks_train, blocks_test = [], []
    for view, lo, hi, max_features, weight in VIEWS:
        prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)
        analyzer = "char" if view == "shape" else "word" if view == "function" else view
        opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True,
                    min_df=2, max_features=max_features, dtype=np.float32)
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        vectorizer = TfidfVectorizer(**opts)
        blocks_train.append(vectorizer.fit_transform([prep(t) for t in train_texts]) * weight)
        blocks_test.append(vectorizer.transform([prep(t) for t in test_texts]) * weight)
    X_train = hstack(blocks_train, format="csr")
    X_test = hstack(blocks_test, format="csr")
    if X_train.shape[0] != len(train_y) or X_test.shape[0] != len(test_texts):
        raise ValueError("SVC feature row count mismatch")
    model = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced")
    model.fit(X_train, train_y)
    probs = safe_prob(model.decision_function(X_test))
    return probs

pool = pd.read_parquet(find("pool_v1.parquet"))
if not {"text", "model"}.issubset(pool.columns):
    raise ValueError("pool_v1.parquet must contain text and model columns")
pool = pool.dropna(subset=["text", "model"]).copy()
pool["text"] = pool["text"].astype(str)
unknown_pool = sorted(set(pool.model) - set(LABELS))
if unknown_pool:
    raise ValueError(f"Unknown pool labels: {unknown_pool}")
fit_texts = train.text.tolist() + pool.text.tolist()
fit_y = np.r_[y, pool.model.map(LABEL_TO_ID).to_numpy(dtype=np.int64)]
P_svc = fit_svc(fit_texts, fit_y, test.text.tolist())
LOGIT += W_SVC * np.log(np.clip(P_svc, 1e-12, 1.0))
del fit_texts, fit_y, P_svc, pool
gc.collect()
print(f"svc complete ({time.time() - T0:.0f}s)", flush=True)


## 2. The LoRA classifiers

Each checkpoint is a LoRA adapter on the **first 24 layers of Qwen3 8B** (or the first 27 of Qwen3 14B). We don't need the rest: style lives in the early and middle layers, while the last layers mostly encode the topic, which is useless here because test prompts are new.

For each passage we:
1. merge the adapter into the weights once (`W <- W + B A * alpha / r`), so no `peft` is needed
2. run the passage and take the mean hidden state at 8 depths (layers 2, 4, 6, 8, 12, 16, 20, 24), each through its own LayerNorm;
3. concatenate them (8 x 4096 numbers) and apply a linear layer to get 18 scores.

The final RMSNorm is switched off (`Identity`) because the checkpoints were trained that way. On a T4 everything runs in fp16, a batch that overflows is redone in bf16.


In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer

def find_base(spec):
    hits=[]
    for cfg_path in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try: cfg=json.load(open(cfg_path))
        except Exception: continue
        if all(cfg.get(k)==v for k,v in spec.items()): hits.append(os.path.dirname(cfg_path))
    if not hits: raise FileNotFoundError(f"Base model not found for {spec}")
    return hits[0]

def load_trunk(base_dir, keep, dtype):
    config=AutoConfig.from_pretrained(base_dir)
    original_layers=config.num_hidden_layers
    if keep <= 0 or keep > original_layers: raise ValueError(f"Invalid keep={keep} for {original_layers} layers")
    config.num_hidden_layers, config.use_cache = keep, False
    if getattr(config, "layer_types", None) is not None: config.layer_types=config.layer_types[:keep]
    hd=getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    gib=2*(config.vocab_size*config.hidden_size + keep*(2*config.hidden_size*config.num_attention_heads*hd + 2*config.hidden_size*config.num_key_value_heads*hd + 3*config.hidden_size*config.intermediate_size))/2**30
    if gib > 13 and torch.cuda.device_count() > 1:
        model=AutoModel.from_pretrained(base_dir, config=config, dtype=dtype, device_map="auto", max_memory={i:"13GiB" for i in range(torch.cuda.device_count())}).eval()
    else:
        model=AutoModel.from_pretrained(base_dir, config=config, dtype=dtype).to("cuda:0").eval()
    model.norm=nn.Identity()
    return model

@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg=json.load(open(os.path.join(adapter_dir,"adapter_config.json")))
    scale=acfg["lora_alpha"]/acfg["r"]
    sd=load_file(os.path.join(adapter_dir,"adapter_model.safetensors"))
    modules=dict(model.named_modules())
    merged=0
    for key,A in sd.items():
        if ".lora_A." not in key: continue
        prefix=key.split(".lora_A.")[0]
        module_name=prefix[prefix.index("layers."):]
        if module_name not in modules: raise KeyError(f"LoRA target missing: {module_name}")
        Bm=sd[key.replace(".lora_A.",".lora_B.")]
        modules[module_name].weight.add_(((Bm.float().to(modules[module_name].weight.device) @ A.float().to(modules[module_name].weight.device))*scale).to(modules[module_name].weight.dtype))
        merged += 1
    if merged == 0: raise ValueError(f"No LoRA tensors merged from {adapter_dir}")

@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    meta=json.load(open(os.path.join(ckpt_dir,"meta.json")))
    if meta.get("labels") != LABELS: raise ValueError("Checkpoint label order differs from production LABELS")
    cfg=meta["cfg"]
    dtype=torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tokenizer=AutoTokenizer.from_pretrained(base_dir); tokenizer.padding_side="right"
    if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
    model=load_trunk(base_dir,cfg["keep"],dtype)
    merge_lora(model,os.path.join(ckpt_dir,"adapter"))
    heads=load_file(os.path.join(ckpt_dir,"heads.safetensors"))
    layers,H=cfg["layers"],model.config.hidden_size
    norms=nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]:v for k,v in heads.items() if k.startswith("norms.")})
    ids=tokenizer(list(texts),add_special_tokens=False,truncation=True,max_length=cfg["max_len"])["input_ids"]
    ids=[x+[tokenizer.eos_token_id] for x in ids]
    order=np.argsort([len(x) for x in ids],kind="stable")
    Z=np.zeros((len(ids),H*len(layers)),np.float32)
    for start in range(0,len(order),bs):
        idx=order[start:start+bs]
        enc=tokenizer.pad({"input_ids":[ids[i] for i in idx]},return_tensors="pt").to(model.embed_tokens.weight.device)
        hs=model(**enc,output_hidden_states=True,use_cache=False).hidden_states
        if not all(torch.isfinite(hs[layer]).all() for layer in layers):
            model.to(torch.bfloat16); hs=model(**enc,output_hidden_states=True,use_cache=False).hidden_states; model.to(dtype)
        if not all(torch.isfinite(hs[layer]).all() for layer in layers): raise FloatingPointError("Non-finite Qwen hidden state")
        mask=enc["attention_mask"].unsqueeze(-1).float().to("cuda:0"); mask[:,0]=0
        den=mask.sum(1).clamp(min=1)
        Z[idx]=torch.cat([norm((hs[layer].to("cuda:0").float()*mask).sum(1)/den) for layer,norm in zip(layers,norms)],-1).cpu().numpy()
    del model; torch.cuda.empty_cache()
    return Z,{k:v.float().numpy() for k,v in heads.items() if not k.startswith("norms.")}


## 3. Optional test-time centering

This stage is feature-gated and **disabled by default**. It is kept here only for a controlled ablation. No clustering is performed when `ENABLE_TWIN=False`, so the baseline cannot be changed accidentally.

In [ ]:
# Qwen plain-head inference; optional centering is isolated behind ENABLE_TWIN.
if ENABLE_TWIN:
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    Xt=TfidfVectorizer(stop_words="english",sublinear_tf=True).fit_transform(test.text.tolist())
    D=np.clip(1-(Xt@Xt.T).toarray(),0,None); np.fill_diagonal(D,0)
    groups=fcluster(linkage(squareform(D,checks=False),method="average"),TWIN["thresh"],criterion="distance") if len(test)>=2 else np.ones(len(test),dtype=int)
    size=np.bincount(groups)[groups]
    twin_rows=(size>=TWIN["min_size"])&(size<=TWIN["max_size"])
    del Xt,D
    print(f"twin enabled: {int(twin_rows.sum())} rows",flush=True)
else:
    groups=None; twin_rows=np.zeros(len(test),dtype=bool)
    print("twin disabled",flush=True)

for ckpt_name,mc in LORAS.items():
    ckpt=os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z,heads=lora_features(ckpt,find_base(mc["base"]),test.text.tolist(),mc["bs"])
    plain=safe_prob(Z@heads["plain.weight"].T+heads["plain.bias"])
    LOGIT += mc["w"]*np.log(np.clip(plain,1e-12,1.0))
    if ENABLE_TWIN and twin_rows.any() and "centered.weight" in heads:
        Zc=Z.copy()
        for group in np.unique(groups[twin_rows]): Zc[groups==group]-=Zc[groups==group].mean(0)
        centered=safe_prob(Zc[twin_rows]@heads["centered.weight"].T+heads["centered.bias"])
        LOGIT[twin_rows] += mc["twin_w"]*np.log(np.clip(centered,1e-12,1.0))
    del Z,heads,plain
    gc.collect()
    print(f"{ckpt_name} complete ({time.time()-T0:.0f}s)",flush=True)


## 4. Controlled blend and submission validation

Only the SVC and Qwen plain heads are active in the production baseline. Calibration and sibling specialists are intentionally not present in this baseline; they must be evaluated as separate experiments before promotion.

In [ ]:
# Final probability normalisation and optional class balancing.
if not np.isfinite(LOGIT).all(): raise FloatingPointError("Non-finite ensemble logits")
P=safe_prob(LOGIT)
if ENABLE_SINKHORN and len(test)>=50:
    target=np.full(K,1.0/K)
    for _ in range(100):
        col=P.sum(0)
        if np.any(col<=0) or not np.isfinite(col).all(): raise FloatingPointError("Invalid Sinkhorn column mass")
        P*=target/col; P/=P.sum(1,keepdims=True)
    if not np.allclose(P.sum(0),target,atol=2e-3): raise FloatingPointError("Sinkhorn did not converge")


In [ ]:
# Fail-fast submission contract.
pred=np.array(LABELS,dtype=object)[P.argmax(1)]
sub=pd.DataFrame({"id":test["id"].astype(str),"model":pred})
if len(sub)!=len(test): raise ValueError("Submission row count mismatch")
if sub.id.duplicated().any(): raise ValueError("Duplicate submission IDs")
if set(sub.id)!=set(test.id.astype(str)): raise ValueError("Submission IDs differ from test IDs")
if set(sub.model)-set(LABELS): raise ValueError("Invalid submission label")
sub.to_parquet("submission.parquet",index=False)
print(f"pipeline={PIPELINE_VERSION} rows={len(sub)} runtime={time.time()-T0:.0f}s",flush=True)
print(sub.model.value_counts().sort_index().to_string(),flush=True)
